# Team total — full_game: the points-per-team model

**Target:** points scored by one team in the game (incl. OT); two rows per game.
Spread (difference), total (sum) and winner derive from the two team predictions.

The full pipeline from `docs/pipeline.md`:

| Step | Data | Output |
|---|---|---|
| 1. Optuna tuning (`scripts/tune_team_points.py`) | walk-forward CV 2019–2023 | `models/team_points_params.json` |
| 2. Out-of-fold predictions per model | CV 2019–2023 | error table, diversity |
| 3. Ensemble: average, top-3, stacking | CV 2019–2023 | weights |
| 4. **Validation** | 2024 (never used before) | pick the final model |
| 5. Market layer: when to trust the edge vs the opening line | fit 2021–2023, check 2024 | bet thresholds |
| 6. **Test, run once** | 2025 | the honest number |

Opening lines only exist from 2021, so the market layer trains on 2021–2023.

In [ ]:
import json

import numpy as np
import pandas as pd
from scipy.optimize import nnls

from canes_cfb import betting, market
from canes_cfb.modeling import SPECS, fit_predict, oof_predictions
from canes_cfb.paths import PROCESSED, RAW, ROOT

params = json.loads((ROOT / "models" / "team_points_params.json").read_text())
features = pd.read_parquet(PROCESSED / "team_games.parquet")
games = pd.read_parquet(RAW / "games.parquet")

base = features[features.completed & ~features.shortened & features.season.between(2016, 2025)]
base = base.reset_index(drop=True)
cv_data = base[base.season <= 2023].reset_index(drop=True)
MODELS = list(SPECS)

## 1. Tuning results (Optuna, walk-forward CV MAE)

In [ ]:
tuning = pd.DataFrame(params).T[["cv_mae_defaults", "cv_mae", "trials"]]
tuning.columns = ["defaults", "tuned", "trials"]
tuning.sort_values("tuned")

## 2. Out-of-fold predictions per model (2019–2023)

In [ ]:
oof = pd.DataFrame({m: oof_predictions(SPECS[m], params[m]["params"], cv_data) for m in MODELS})
scored = oof.notna().all(axis=1)
y_cv = cv_data.points[scored]


def mae(pred, y):
    return float(np.abs(np.asarray(y) - np.asarray(pred)).mean())


single = pd.Series({m: mae(oof.loc[scored, m], y_cv) for m in MODELS}).sort_values()
single.round(3)

Error correlation between models. An ensemble only helps if the models make
*different* mistakes.

In [ ]:
errors = oof[scored].sub(y_cv, axis=0)
errors.corr().round(2)

## 3. Ensembles

In [ ]:
top3 = list(single.index[:3])
weights, _ = nnls(oof.loc[scored, MODELS].to_numpy(), y_cv.to_numpy())
stack_w = pd.Series(weights, index=MODELS)


def combine(preds: pd.DataFrame) -> dict[str, pd.Series]:
    return {
        "average (all)": preds[MODELS].mean(axis=1),
        "average (top 3)": preds[top3].mean(axis=1),
        "stacking (NNLS)": preds[MODELS] @ stack_w,
    }


ensembles_cv = {k: mae(v[scored], y_cv) for k, v in combine(oof).items()}
print("stacking weights:", stack_w.round(3).to_dict())
pd.Series(ensembles_cv).round(3)

Stacking weights are fit on these same out-of-fold rows, so its CV number is a little
optimistic. That's why the choice is made on 2024 next.

## 4. Validation: 2024

In [ ]:
train_v = base[base.season <= 2023]
valid = base[base.season == 2024].copy()
val_preds = pd.DataFrame(
    {m: fit_predict(SPECS[m], params[m]["params"], train_v, valid) for m in MODELS},
    index=valid.index,
)
candidates = {**{m: val_preds[m] for m in MODELS}, **combine(val_preds)}
validation = pd.Series({k: mae(v, valid.points) for k, v in candidates.items()})
validation["market: opening line"] = mae(valid.market_points_open, valid.points)
validation["market: closing line"] = mae(valid.market_points, valid.points)
validation.sort_values().round(3)

In [ ]:
model_only = validation.drop(["market: opening line", "market: closing line"])
FINAL = model_only.idxmin()
recipe = {
    "final": FINAL,
    "models": MODELS,
    "top3": top3,
    "stack_weights": stack_w.round(6).to_dict(),
}
(ROOT / "models" / "team_points_final.json").write_text(json.dumps(recipe, indent=2))
print("final model:", FINAL)

## 5. Market layer: when to trust the edge against the opening line

Game-level rows: out-of-fold predictions for 2021–2023 (fit) and 2024 (check). A tiny
logistic regression turns the edge vs. the opener into P(side wins).

In [ ]:
def final_prediction(preds: pd.DataFrame) -> pd.Series:
    return candidates_fn(preds)[FINAL]


def candidates_fn(preds: pd.DataFrame) -> dict[str, pd.Series]:
    return {**{m: preds[m] for m in MODELS}, **combine(preds)}


def game_rows(rows: pd.DataFrame, pred: pd.Series) -> pd.DataFrame:
    team_rows = rows.assign(pred=pred.to_numpy())
    keep = [
        "game_id",
        "team_id",
        "pred",
        "season",
        "season_type",
        "week",
        "spread_open",
        "spread_close",
        "total_open",
        "total_close",
    ]
    return betting.to_games(team_rows[keep], games)


cv_rows = cv_data[scored]
g_fit = game_rows(cv_rows, final_prediction(oof[scored]))
g_fit = g_fit[g_fit.season >= 2021]
g_val = game_rows(valid, final_prediction(val_preds))
print(f"fit games (2021–2023): {len(g_fit)} | validation games (2024): {len(g_val)}")

In [ ]:
layers, layer_results = {}, {}
for kind in ("spread", "total"):
    layers[kind] = market.fit(g_fit, kind)
    p_val = market.side_probability(layers[kind], g_val, kind)
    layer_results[kind] = market.grade_by_confidence(p_val, g_val, kind)
    print(
        kind,
        "- logistic coefficients:",
        dict(zip(market.INPUTS, layers[kind][-1].coef_[0].round(3), strict=False)),
    )
    display(layer_results[kind].round(2))

For comparison, the raw edge rule on 2024 (bet whenever the edge clears N points):

In [ ]:
raw_rule = {
    "spread": betting.grade(
        betting.spread_edge(g_val, "spread_open"),
        betting.spread_result(g_val, "spread_open"),
        betting.spread_clv(betting.spread_edge(g_val, "spread_open"), g_val),
    ),
    "total": betting.grade(
        betting.total_edge(g_val, "total_open"),
        betting.total_result(g_val, "total_open"),
        betting.total_clv(betting.total_edge(g_val, "total_open"), g_val),
    ),
}
for kind, table in raw_rule.items():
    print(kind)
    display(table.round(2))

## 6. Test: 2025 (run once)

Everything is frozen: the tuned parameters, the ensemble recipe (`FINAL`), and the
market layer. Base models retrain on 2016–2024; the market layer refits on 2021–2024
(out-of-fold 2021–2023 + validation 2024). Then 2025 is predicted and graded, once.

In [ ]:
train_t = base[base.season <= 2024]
test = base[base.season == 2025].copy()
test_preds = pd.DataFrame(
    {m: fit_predict(SPECS[m], params[m]["params"], train_t, test) for m in MODELS},
    index=test.index,
)
test["pred"] = final_prediction(test_preds)
test_mae = pd.Series(
    {
        f"model ({FINAL})": mae(test.pred, test.points),
        "market: opening line": mae(test.market_points_open, test.points),
        "market: closing line": mae(test.market_points, test.points),
        "ratings formula": mae(test.exp_points, test.points),
    }
)
test_mae.round(3)

In [ ]:
g_test = game_rows(test, test.pred)
g_refit = pd.concat([g_fit, g_val], ignore_index=True)
test_results = {}
for kind in ("spread", "total"):
    layer = market.fit(g_refit, kind)
    p_test = market.side_probability(layer, g_test, kind)
    test_results[kind] = market.grade_by_confidence(p_test, g_test, kind)
    print(f"{kind}: 2025 test, market layer")
    display(test_results[kind].round(2))
print("break-even at -110: 52.38%")

# The weekly predictions refit the market layer on every graded season with an opener.
pd.concat([g_refit, g_test], ignore_index=True).to_parquet(
    PROCESSED / "market_layer_games.parquet", index=False
)

In [ ]:
for kind, (edge_fn, result_fn, clv_fn, line) in {
    "spread": (betting.spread_edge, betting.spread_result, betting.spread_clv, "spread"),
    "total": (betting.total_edge, betting.total_result, betting.total_clv, "total"),
}.items():
    for when in ("open", "close"):
        edge = edge_fn(g_test, f"{line}_{when}")
        clv = clv_fn(edge, g_test) if when == "open" else None
        print(f"{kind} vs {when}: raw edge rule, 2025 test")
        display(betting.grade(edge, result_fn(g_test, f"{line}_{when}"), clv).round(2))

## 7. Conclusions (test run once, 2026-09-23)

**Accuracy (MAE, points per team)**

| | CV 2019–23 | Validation 2024 | Test 2025 |
|---|---|---|---|
| Final model (random forest, chosen on 2024) | 9.28 | 9.08 | **8.94** |
| Best CV single model (XGBoost) / stacking ensemble | 9.24 / 9.20 | 9.13 / 9.10 | |
| Market: opening line | | 8.87 | 8.70 |
| Market: closing line | | 8.86 | 8.59 |

CV column = pooled out-of-fold MAE (section 2); Optuna's per-season-mean MAE is ~0.02
higher. Tuning and ensembling help a little (best single model 9.30 → 9.26 per-season
mean; stacking 9.20 pooled), but every model is within
~0.05 of the others and highly correlated (error correlation ≥ 0.98). **The market is
still more accurate**, by ~0.25 points per team. The final pick (random forest) won
2024 by a margin that's within noise.

**Betting, 2025 test (−110, break-even 52.38%)**
- Market layer, spreads: 47–49%. It doesn't work.
- Market layer, totals: 55.7% at p ≥ 0.5. **Not credible as model skill:** its edge
  coefficient is negative, and it mostly bets unders on high totals. 2025 happened to be
  an under year (53.7% of games; 57.4% of high totals), while in 2022–2023 that rule was
  ~50%. The market layer is dropped from the weekly card.
- Raw model edge vs the opening line:
  - spreads: 50.9% overall. 55.1% at ≥ 4 pts in 2025, but 45.9% in 2024, so noise.
  - **totals at ≥ 4 pts: 57.8% (2024, n = 213) and 55.2% (2025, n = 201).** The only rule
    above break-even in both held-out years. Still within ~1 standard error of
    break-even, and the threshold was chosen after looking.
- **CLV is positive every year** and grows with the edge (+0.3 → +1.0 pts vs the opener
  in 2025). The market consistently moves toward the model. That's the most robust
  signal we have.

**Verdict:** not ready for real money. Paper-track the pre-registered rule (totals,
|edge| ≥ 4 vs the opening total) through 2026 with `scripts/predict_week.py`, and log
results and CLV weekly.